# ELG Example Notebook

이 노트북은 `elg` 패키지의 공개 API를 **설명 → 코드** 순서로 하나씩 보여준다.
각 셀은 독립적으로 읽기 쉽게 구성했고, 구조체 / 렌더링 / mutation / codec / normalize / metrics / sampler 순서로 정리했다.

## 1. 전체 공개 API import

먼저 이후 예시들에서 사용할 `elg` 공개 API를 한 번에 import한다.

In [3]:
from elg import (
    AtomicNode,
    AtomicSource,
    AtomicType,
    Hypothesis,
    LogicalNode,
    LogicalOp,
    MutationSample,
    Path,
    RelationNode,
    RelationType,
    count_atomics,
    count_logicals,
    count_nodes,
    count_relations,
    fingerprint,
    generate_mutation_candidates,
    get_node_at_path,
    hypothesis_from_dict,
    hypothesis_from_json,
    hypothesis_to_json,
    iter_paths,
    mutate_append_child,
    mutate_logical_operator,
    mutate_relation_type,
    mutate_remove_child,
    mutate_replace_child,
    mutate_replace_subtree,
    mutate_unwrap_not,
    mutate_wrap_not,
    node_from_dict,
    node_to_dict,
    normalize_hypothesis,
    normalize_node,
    render_pretty,
    render_tree,
    replace_at_path,
    sample_mutation,
    tree_depth,
)
import random

## 2. Enum 타입들

`AtomicType`, `AtomicSource`, `LogicalOp`, `RelationType`은 ELG의 닫힌 vocabulary를 나타낸다.

In [ ]:
print(list(AtomicType))
print(list(AtomicSource))
print(list(LogicalOp))
print(list(RelationType))

## 3. AtomicNode 생성

`AtomicNode`는 더 이상 ELG 내부에서 분해하지 않는 opaque leaf proposition이다.

In [ ]:
primitive_atomic = AtomicNode("FUNDING_FEE > 0", type=AtomicType.BOOLEAN, source=AtomicSource.PRIMITIVE)
semantic_atomic = AtomicNode("시장 상태가 불안정하다", type=AtomicType.ABSTRACT, source=AtomicSource.SEMANTIC)

print(primitive_atomic)
print(semantic_atomic)
print(primitive_atomic.kind)

## 4. LogicalNode 생성

`LogicalNode`는 `AND`, `OR`, `NOT` 같은 논리 연산자를 가진다.

In [ ]:
and_node = LogicalNode(LogicalOp.AND, [
    AtomicNode("FUNDING_FEE > 0"),
    AtomicNode("CLOSE > SMA_20"),
])
not_node = LogicalNode(LogicalOp.NOT, [AtomicNode("RETURN_5M > 0")])

print(and_node)
print(not_node)
print(and_node.kind, and_node.op)

## 5. RelationNode 생성

`RelationNode`는 조건과 결과를 연결하는 관계 노드다.

In [ ]:
relation = RelationNode(
    RelationType.IMPLIES,
    [and_node, AtomicNode("RETURN_5M > 0")],
)

print(relation)
print(relation.kind)
print("condition =", relation.condition)
print("target =", relation.target)

## 6. Hypothesis 생성

`Hypothesis`는 root node 하나를 감싸는 최상위 wrapper다.

In [ ]:
hypothesis = Hypothesis(root=relation, params={"name": "funding-trend hypothesis"})
print(hypothesis)

## 7. node_to_dict

`node_to_dict`는 단일 node를 dict로 직렬화한다.

In [ ]:
print(node_to_dict(and_node))
print(node_to_dict(relation))

## 8. hypothesis_to_json

`hypothesis_to_json`은 hypothesis 전체를 JSON 문자열로 만든다.

In [ ]:
payload_json = hypothesis_to_json(hypothesis)
print(payload_json)

## 9. node_from_dict

`node_from_dict`는 `kind`를 기준으로 atomic / logical / relation node를 복원한다.

In [ ]:
node_payload = {
    "kind": "logical",
    "op": "OR",
    "inputs": [
        {"kind": "atomic", "name": "A", "type": "abstract", "source": "semantic", "params": {}},
        {"kind": "atomic", "name": "B", "type": "abstract", "source": "semantic", "params": {}},
    ],
    "params": {},
}
restored_node = node_from_dict(node_payload)
print(restored_node)
print(render_pretty(restored_node))

## 10. hypothesis_from_dict

dict payload를 다시 `Hypothesis` 객체로 복원할 수 있다.

In [ ]:
hypothesis_payload = hypothesis.to_dict()
restored_hypothesis = hypothesis_from_dict(hypothesis_payload)
print(restored_hypothesis)
print(render_pretty(restored_hypothesis))

## 11. hypothesis_from_json

JSON 문자열로부터 hypothesis를 복원한다.

In [ ]:
restored_from_json = hypothesis_from_json(payload_json)
print(restored_from_json)
print(render_pretty(restored_from_json))

## 12. render_pretty

`render_pretty`는 사람이 읽기 좋은 중첩 표현으로 ELG를 보여준다.

In [ ]:
print(render_pretty(hypothesis))

## 13. render_tree

`render_tree`는 ASCII tree 형태로 ELG 구조를 시각화한다.

In [ ]:
print(render_tree(hypothesis))

## 14. normalize_node

단일 node를 정규화한다. 예를 들어 중복 child 제거, child 정렬, double negation 제거 같은 처리를 한다.

In [ ]:
unnormalized_node = LogicalNode("AND", [
    AtomicNode("B"),
    AtomicNode("A"),
    AtomicNode("A"),
])
normalized_node = normalize_node(unnormalized_node)
print(render_pretty(unnormalized_node))
print('---')
print(render_pretty(normalized_node))

## 15. normalize_hypothesis

전체 hypothesis를 정규화한다.

In [ ]:
unnormalized_hypothesis = Hypothesis(
    root=LogicalNode("NOT", [LogicalNode("NOT", [AtomicNode("A")])])
)
normalized_hypothesis = normalize_hypothesis(unnormalized_hypothesis)
print(render_pretty(unnormalized_hypothesis))
print('---')
print(render_pretty(normalized_hypothesis))

## 16. 구조 메트릭: count_nodes, tree_depth, count_atomics, count_logicals, count_relations

ELG의 구조적 복잡도를 정량화하는 기본 함수들이다.

In [ ]:
print("count_nodes =", count_nodes(hypothesis))
print("tree_depth =", tree_depth(hypothesis))
print("count_atomics =", count_atomics(hypothesis))
print("count_logicals =", count_logicals(hypothesis))
print("count_relations =", count_relations(hypothesis))

## 17. fingerprint

`fingerprint`는 정규화된 구조를 기반으로 해시를 만든다. 구조적으로 같은 가설은 같은 fingerprint를 갖는다.

In [ ]:
h1 = Hypothesis(root=LogicalNode("AND", [AtomicNode("A"), AtomicNode("B")]))
h2 = Hypothesis(root=LogicalNode("AND", [AtomicNode("B"), AtomicNode("A")]))

print(fingerprint(h1))
print(fingerprint(h2))
print("same fingerprint:", fingerprint(h1) == fingerprint(h2))

## 18. Path 타입

`Path`는 트리 안의 특정 위치를 가리키는 tuple 기반 주소다.

- `()` = root
- `(0,)` = 첫 번째 child
- `(0, 1)` = root의 첫 child의 두 번째 child

In [ ]:
root_path: Path = ()
condition_path: Path = (0,)
second_condition_child_path: Path = (0, 1)

a = root_path, condition_path, second_condition_child_path
print(a)

## 19. get_node_at_path

주어진 path의 node를 가져온다.

In [ ]:
print(get_node_at_path(hypothesis, ()))
print(get_node_at_path(hypothesis, (0,)))
print(get_node_at_path(hypothesis, (0, 1)))

## 20. iter_paths

현재 hypothesis에서 접근 가능한 모든 path를 순회한다.

In [ ]:
print(iter_paths(hypothesis))

## 21. replace_at_path

지정한 path의 subtree를 새 node로 교체한다. 원본 hypothesis는 바뀌지 않는다.

In [ ]:
replaced = replace_at_path(hypothesis, (0, 1), AtomicNode("OPEN_INTEREST_CHANGE > 0"))
print(render_pretty(replaced))
print('--- original ---')
print(render_pretty(hypothesis))

## 22. mutate_replace_subtree

`replace_at_path`의 mutation-friendly wrapper로, subtree 전체를 새 구조로 바꾼다.

In [ ]:
subtree_mutated = mutate_replace_subtree(
    hypothesis,
    (0,),
    LogicalNode("OR", [AtomicNode("VOLATILITY_HIGH"), AtomicNode("FUNDING_FEE > 0")]),
)
print(render_pretty(subtree_mutated))

## 23. mutate_replace_child

논리 노드나 관계 노드의 특정 child만 교체한다.

In [ ]:
child_mutated = mutate_replace_child(hypothesis, (0,), 1, AtomicNode("VOLUME > AVG_VOLUME"))
print(render_pretty(child_mutated))

## 24. mutate_logical_operator

`AND -> OR`, `OR -> AND` 같은 logical operator 변경을 수행한다.

In [ ]:
logical_mutated = mutate_logical_operator(hypothesis, (0,), "OR")
print(render_pretty(logical_mutated))

## 25. mutate_relation_type

relation 종류를 바꾼다.

In [ ]:
relation_mutated = mutate_relation_type(hypothesis, (), "SUPPORT")
print(render_pretty(relation_mutated))

## 26. mutate_wrap_not

지정한 path의 node를 `NOT(node)`로 감싼다.

In [ ]:
wrapped_not = mutate_wrap_not(hypothesis, (1,))
print(render_pretty(wrapped_not))

## 27. mutate_unwrap_not

`NOT(A)`를 다시 `A`로 푼다.

In [ ]:
unwrapped = mutate_unwrap_not(wrapped_not, (1,))
print(render_pretty(unwrapped))

## 28. mutate_append_child

`AND/OR` 노드에 child를 하나 더 추가한다.

In [ ]:
appended = mutate_append_child(hypothesis, (0,), AtomicNode("VOLUME > AVG_VOLUME"))
print(render_pretty(appended))

## 29. mutate_remove_child

`AND/OR` 노드에서 특정 child를 제거한다. 최소 arity를 깨면 예외가 난다.

In [ ]:
removed = mutate_remove_child(appended, (0,), 1)
print(render_pretty(removed))

## 30. MutationSample

샘플러는 결과를 `MutationSample`로 반환한다. 안에는 어떤 mutation이 선택됐는지 메타데이터가 들어 있다.

In [ ]:
seeded_sample = MutationSample(
    operation="manual_demo",
    path=(0,),
    result=hypothesis,
    details={"note": "demo object"},
)
print(seeded_sample)

## 31. generate_mutation_candidates

현재 hypothesis에서 가능한 legal mutation 후보들을 전부 생성한다.

In [ ]:
candidates = generate_mutation_candidates(hypothesis, atomic_pool=["X", "D"])
print("candidate count =", len(candidates))
for item in candidates[:5]:
    print(item.operation, item.path, item.details)

## 32. sample_mutation

후보들 중 하나를 랜덤하게 샘플링한다. `random.Random(seed)`를 넣으면 재현 가능하다.

In [ ]:
sampled = sample_mutation(hypothesis, rng=random.Random(7), atomic_pool=["X", "D"])
print(sampled.operation)
print(sampled.path)
print(sampled.details)
print(render_pretty(sampled.result))

## 33. 전체 흐름 예시

마지막으로 ELG를 한 번에 생성하고, 렌더링하고, mutate하고, serialize하고, normalize/fingerprint를 계산하는 전체 흐름을 보여준다.

In [ ]:
base = Hypothesis(
    root=RelationNode(
        "IMPLIES",
        [
            LogicalNode("AND", [
                AtomicNode("FUNDING_FEE > 0"),
                AtomicNode("CLOSE > SMA_20"),
            ]),
            AtomicNode("RETURN_5M > 0"),
        ],
    ),
    params={"name": "end-to-end demo"},
)

print('PRETTY')
print(render_pretty(base))
print('---')
print('TREE')
print(render_tree(base))
print('---')
mutated = sample_mutation(base, rng=random.Random(3), atomic_pool=["VOLATILITY_HIGH", "OPEN_INTEREST_CHANGE > 0"])
print('SAMPLED MUTATION =', mutated.operation, mutated.path, mutated.details)
print(render_pretty(mutated.result))
print('---')
json_payload = hypothesis_to_json(mutated.result)
print(json_payload)
print('---')
restored = hypothesis_from_json(json_payload)
print('fingerprint =', fingerprint(restored))
print('normalized =')
print(render_pretty(normalize_hypothesis(restored)))
print('metrics =', count_nodes(restored), tree_depth(restored), count_atomics(restored), count_logicals(restored), count_relations(restored))

## 34. LLMClient with OpenRouter

이 예시는 `hypoevolve.llm.LLMClient`를 **OpenRouter**와 함께 사용하는 방법을 보여준다.

- backend: OpenRouter (OpenAI-compatible)
- model: `deepseek/deepseek-v3.2`

> 보안상 **실제 API 키를 노트북에 직접 저장하지 말고**, 환경변수로 주입하는 방식을 사용한다.

In [4]:
import os
from hypoevolve import LLMClient, LLMConfig

# 권장: 셸에서 먼저 export OPENAI_API_KEY=... 해두기
# 또는 이 셀에서 임시로 설정하기 (노트북 저장 전 제거 권장)
# os.environ["OPENAI_API_KEY"] = "sk-or-..."
API_KEY = "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"

llm_config = LLMConfig(
    model="deepseek/deepseek-v3.2",
    api_base="https://openrouter.ai/api/v1",
    api_key=API_KEY,
    temperature=0.2,
    max_tokens=1000,
    timeout=60,
    retries=2,
    retry_delay=1.0,
)

client = LLMClient(llm_config)
print(llm_config)

LLMConfig(model='deepseek/deepseek-v3.2', temperature=0.2, max_tokens=1000, api_key='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', api_base='https://openrouter.ai/api/v1', timeout=60, retries=2, retry_delay=1.0)


## 35. generate_text 예시

`generate_text()`는 system prompt와 user prompt를 받아 일반 텍스트 응답을 반환한다.

In [4]:
system_prompt = "You are a concise research assistant."
user_prompt = "Summarize why structured hypothesis mutation can be more stable than raw natural-language mutation in 3 bullets."

# 실제 호출 예시
response = client.generate_text(system_prompt, user_prompt)
print(response)

print("client.generate_text(system_prompt, user_prompt)")

- **Reduced Ambiguity:** Structured formats enforce clear, testable components (e.g., variables, relationships), minimizing the misinterpretation and inconsistency common in free-text natural language.
- **Controlled Variation:** Mutations can be applied systematically to specific elements (like parameters or operators), ensuring logical coherence and preventing nonsensical or irrelevant changes.
- **Automation & Consistency:** Structured hypotheses are machine-readable, enabling reliable automated generation, evaluation, and iteration, which reduces human error and bias over time.
client.generate_text(system_prompt, user_prompt)


## 36. generate_json 예시

`generate_json()`는 LLM에게 JSON만 반환하도록 지시하고, 그 응답을 Python `dict`로 파싱한다.

실전에서는 출력 계약을 분명히 적는 것이 중요하다.

In [6]:
system_prompt = "Return only valid JSON."
user_prompt = """
Return a JSON object with keys:
- score: float between 0 and 1
- rationale: short string
for the hypothesis: 'If funding fee is positive then short-term returns are positive.'
"""

# 실제 호출 예시
payload = client.generate_json(system_prompt, user_prompt)
print(payload)

print("client.generate_json(system_prompt, user_prompt)")

{'score': 0.3, 'rationale': 'Funding fee direction does not guarantee short-term return direction'}
client.generate_json(system_prompt, user_prompt)


In [7]:
payload

{'score': 0.3,
 'rationale': 'Funding fee direction does not guarantee short-term return direction'}

## 40. 실제 LLM parser 호출 예시

이 셀은 `LLMClient`와 `llm_parse_hypothesis()`를 실제로 연결해서, 자연어 가설을 ELG로 변환하는 예시다.

- backend: OpenRouter
- model: `deepseek/deepseek-v3.2`

> 실행 전에 `OPENAI_API_KEY` 환경변수가 설정되어 있어야 한다.

In [3]:
from hypoevolve import load_prompt
from hypoevolve import LLMClient, LLMConfig, llm_parse_hypothesis
from elg import render_pretty, render_tree

API_KEY = "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"

llm_config = LLMConfig(
    model="deepseek/deepseek-v3.2",
    api_base="https://openrouter.ai/api/v1",
    api_key=API_KEY,  # 환경변수 OPENAI_API_KEY 사용
    temperature=0.2,
    max_tokens=1200,
    timeout=60,
    retries=2,
    retry_delay=1.0,
)

client = LLMClient(llm_config)

## 41. 자연어 가설을 ELG로 파싱해보기

아래 셀은 실제로 자연어 가설 문자열을 LLM parser로 보내고, 반환된 ELG를 pretty/tree 형태로 확인하는 예시다.

In [4]:
hypothesis_text = "The statistical dependency suggests that sharp downward accelerations in BTCUSDT price, indicated by the NewLow signal derived from the normalized 10-day close momentum, are systematically followed by significant jumps in the transformed high ETHUSDT price series. This relationship likely captures a 'squeeze-and-release' dynamic where extreme negative momentum (a new low) creates oversold conditions, triggering a subsequent short-covering rally or reversal spike that manifests as a jump in the exponential z-score of recent highs. The hypothesis predicts that markets exhibiting this pattern have an asymmetric volatility response to new lows, where the initial sharp sell-off reliably forces a transient, high-magnitude recovery spike within the same period."

# 실제 호출
parsed_hypothesis = llm_parse_hypothesis(hypothesis_text, llm=client, retries=2)

print("=== PRETTY ===")
print(render_pretty(parsed_hypothesis))
print()
print("=== TREE ===")
print(render_tree(parsed_hypothesis))
print()
print("=== RAW DICT ===")
print(parsed_hypothesis.to_dict())

=== PRETTY ===
IMPLIES(
  sharp downward accelerations in BTCUSDT price, indicated by the NewLow signal derived from the normalized 10-day close momentum,
  significant jumps in the transformed high ETHUSDT price series
)

=== TREE ===
Hypothesis
└── IMPLIES
    ├── sharp downward accelerations in BTCUSDT price, indicated by the NewLow signal derived from the normalized 10-day close momentum
    └── significant jumps in the transformed high ETHUSDT price series

=== RAW DICT ===
{'root': {'kind': 'relation', 'type': 'IMPLIES', 'inputs': [{'kind': 'atomic', 'name': 'sharp downward accelerations in BTCUSDT price, indicated by the NewLow signal derived from the normalized 10-day close momentum', 'type': 'abstract', 'source': 'semantic', 'params': {}}, {'kind': 'atomic', 'name': 'significant jumps in the transformed high ETHUSDT price series', 'type': 'abstract', 'source': 'semantic', 'params': {}}], 'params': {}}, 'params': {}}


In [ ]:
try:
    parsed_hypothesis = llm_parse_hypothesis(hypothesis_text, llm=client, retries=2)
except Exception as e:
    print(type(e).__name__, e)
    if hasattr(e, "errors"):
        print("---- detailed errors ----")
        for item in e.errors:
            print(item)

ParseError Failed to convert natural-language hypothesis to ELG via LLM
---- detailed errors ----
attempt 1: Unsupported logical operator: None
attempt 2: Unsupported logical operator: None
attempt 3: Unsupported logical operator: None


In [ ]:
from hypoevolve.parser import PARSER_SYSTEM_PROMPT, PARSER_RETRY_PROMPT

system_prompt = PARSER_SYSTEM_PROMPT
user_prompt = f"Convert this natural-language hypothesis into ELG JSON root node:\n\n{hypothesis_text}"

raw = client.generate_text(system_prompt, user_prompt)
print(raw)

payload = client.generate_json(system_prompt, user_prompt)
print(payload)

## 42. semantic ELG를 measurable ELG로 변환하기

이 셀은 이미 만들어진 semantic ELG hypothesis를 받아서, `llm_make_hypothesis_measurable()`를 통해 더 measurable 한 ELG로 바꾸는 예시다.

핵심 아이디어:
- relation / logical 구조는 최대한 유지
- atomic proposition을 더 계산 가능한 형태로 구체화

In [1]:
from hypoevolve import llm_make_hypothesis_measurable
from elg import AtomicNode, Hypothesis, LogicalNode, RelationNode, render_pretty, render_tree

semantic_hypothesis = Hypothesis(
    root=RelationNode(
        "IMPLIES",
        [
            AtomicNode(
                "sharp downward accelerations in BTCUSDT price indicated by NewLow signal from normalized close momentum"
            ),
            AtomicNode(
                "significant jumps in high ETHUSDT price series"
            ),
        ],
    )
)

print("=== SEMANTIC ===")
print(render_pretty(semantic_hypothesis))

=== SEMANTIC ===
IMPLIES(
  sharp downward accelerations in BTCUSDT price indicated by NewLow signal from normalized close momentum,
  significant jumps in high ETHUSDT price series
)


## 43. measurable 변환 실제 호출 예시

이 셀은 실제로 LLM을 호출해 measurable ELG를 생성하는 예시다.

In [4]:
measurable_hypothesis = llm_make_hypothesis_measurable(
    semantic_hypothesis,
    llm=client,
    retries=2,
)

print("=== MEASURABLE / PRETTY ===")
print(render_pretty(measurable_hypothesis))
print()
print("=== MEASURABLE / TREE ===")
print(render_tree(measurable_hypothesis))
print()
print("=== MEASURABLE / RAW DICT ===")
print(measurable_hypothesis.to_dict())

=== MEASURABLE / PRETTY ===
IMPLIES(
  AND(
    BTCUSDT_NEW_LOW_SIGNAL_W12@t == True,
    BTCUSDT_ZSCORE_CLOSE_MOMENTUM_W12@t < -2.0
  ),
  ETHUSDT_ZSCORE_HIGH_JUMP_W12@t+1 > 1.5
)

=== MEASURABLE / TREE ===
Hypothesis
└── IMPLIES
    ├── AND
    │   ├── BTCUSDT_NEW_LOW_SIGNAL_W12@t == True
    │   └── BTCUSDT_ZSCORE_CLOSE_MOMENTUM_W12@t < -2.0
    └── ETHUSDT_ZSCORE_HIGH_JUMP_W12@t+1 > 1.5

=== MEASURABLE / RAW DICT ===
{'root': {'kind': 'relation', 'type': 'IMPLIES', 'inputs': [{'kind': 'logical', 'op': 'AND', 'inputs': [{'kind': 'atomic', 'name': 'BTCUSDT_NEW_LOW_SIGNAL_W12@t == True', 'type': 'boolean', 'source': 'primitive', 'params': {}}, {'kind': 'atomic', 'name': 'BTCUSDT_ZSCORE_CLOSE_MOMENTUM_W12@t < -2.0', 'type': 'boolean', 'source': 'primitive', 'params': {}}], 'params': {}}, {'kind': 'atomic', 'name': 'ETHUSDT_ZSCORE_HIGH_JUMP_W12@t+1 > 1.5', 'type': 'boolean', 'source': 'primitive', 'params': {}}], 'params': {}}, 'params': {}}


## 44. ELG를 자연어 문장으로 바꾸기

이 셀은 이미 구성된 ELG hypothesis를 받아서, `llm_hypothesis_to_natural_language()`로 사람이 읽기 좋은 자연어 문장으로 바꾸는 예시다.

In [5]:
from hypoevolve import llm_hypothesis_to_natural_language

natural_language_text = llm_hypothesis_to_natural_language(
    measurable_hypothesis,
    llm=client,
    retries=2,
)

print(natural_language_text)

If BTCUSDT has a new low signal in a 12-step window at time t and BTCUSDT's z-score close momentum in a 12-step window at time t is less than -2.0, then ETHUSDT's z-score high jump in a 12-step window at time t+1 is greater than 1.5.


## 45. LocalSubprocessExecutor 사용 예시

이 셀은 `LocalSubprocessExecutor`를 사용해서 LLM이 생성했다고 가정한 Python 코드를 임시 작업 디렉토리에서 실행하고, 결과를 회수하는 예시다.

In [ ]:
from hypoevolve import LocalSubprocessExecutor

executor = LocalSubprocessExecutor(timeout=10, cleanup=False)

result = executor.execute(
    """
print('hello from generated code')
value = 21 * 2
print(f'value={value}')
"""
)

print("stdout:")
print(result.stdout)
print("stderr:")
print(result.stderr)
print("exit_code:", result.exit_code)
print("timed_out:", result.timed_out)
print("duration_sec:", result.duration_sec)
print("work_dir:", result.work_dir)

## 46. parquet 기반 DatasetSchema 예시

이 셀은 여러 parquet 파일을 entity별로 묶어서 `DatasetSchema`를 만드는 예시다.

가정:
- `BTCUSDT.parquet`
- `ETHUSDT.parquet`
같이 자산별 parquet 파일이 따로 있음

In [ ]:
from hypoevolve import ColumnSpec, DataFile, DatasetSchema, DatasetAccessor, IndexSpec

schema = DatasetSchema(
    files=[
        DataFile(entity="BTCUSDT", path="data/BTCUSDT.parquet"),
        DataFile(entity="ETHUSDT", path="data/ETHUSDT.parquet"),
    ],
    index=IndexSpec(name="close_time", dtype="datetime64[us]"),
    columns=[
        ColumnSpec(name="open", description="open price"),
        ColumnSpec(name="high", description="high price"),
        ColumnSpec(name="low", description="low price"),
        ColumnSpec(name="close", description="close price"),
        ColumnSpec(name="volume", description="traded volume"),
        ColumnSpec(name="funding_fee", description="funding fee at each timestamp"),
    ],
    description="Per-asset OHLCV parquet dataset",
)

accessor = DatasetAccessor(schema)
print(accessor.summary())

## 47. YAML 기반 DatasetSchema 로딩 예시

실전에서는 dataset schema를 코드에 직접 쓰기보다 YAML 파일로 관리하는 편이 더 편하다.

In [6]:
from hypoevolve import load_dataset_schema
from hypoevolve import DatasetAccessor

# 예시 파일 형태:
# dataset.yaml
# description: Per-asset OHLCV parquet dataset
# index:
#   name: close_time
#   dtype: datetime64[us]
# files:
#   -
#     entity: BTCUSDT
#     path: data/BTCUSDT.parquet
#   -
#     entity: ETHUSDT
#     path: data/ETHUSDT.parquet
# columns:
#   -
#     name: close
#     description: close price
#   -
#     name: funding_fee
#     description: funding fee at each timestamp

loaded_schema = load_dataset_schema("dataset.yaml")
accessor = DatasetAccessor(loaded_schema)
print(accessor.summary())

print("load_dataset_schema('dataset.yaml')")

{'description': 'Per-asset Binance perpetual futures OHLCV-style parquet dataset for BTCUSDT, DOGEUSDT, and XRPUSDT. The parquet files are indexed by close_time at 5-minute frequency.', 'index_name': 'close_time', 'index_dtype': 'datetime64[us]', 'entities': ['BTCUSDT', 'DOGEUSDT', 'XRPUSDT'], 'columns': ['OPEN', 'HIGH', 'LOW', 'CLOSE', 'VOLUME', 'TAKER_BUY_VOLUME', 'TAKER_SELL_VOLUME', 'ORDER_FLOW_IMBALANCE', 'PREMIUM_INDEX_OPEN', 'PREMIUM_INDEX_HIGH', 'PREMIUM_INDEX_LOW', 'PREMIUM_INDEX_CLOSE', 'FUNDING_SCORE'], 'column_descriptions': {'OPEN': 'opening price for the interval', 'HIGH': 'highest traded price during the interval', 'LOW': 'lowest traded price during the interval', 'CLOSE': 'closing price for the interval', 'VOLUME': 'traded volume during the interval', 'TAKER_BUY_VOLUME': 'taker buy volume during the interval', 'TAKER_SELL_VOLUME': 'taker sell volume during the interval', 'ORDER_FLOW_IMBALANCE': 'imbalance between taker buy and taker sell activity', 'PREMIUM_INDEX_OPEN':

## 48. DatasetAccessor 사용 예시

이 셀은 accessor가 어떤 helper를 제공하는지 보여준다.

In [2]:
print(accessor.entities())
print(accessor.column_names())
print(accessor.column_descriptions())

# 실제 parquet 파일이 있을 때:
rows = accessor.head("BTCUSDT", n=5)
print(rows)

['BTCUSDT', 'XRPUSDT']
['open', 'high', 'low', 'close', 'volume', 'funding_fee', 'open_interest']
{'open': 'open price', 'high': 'high price', 'low': 'low price', 'close': 'close price', 'volume': 'traded volume', 'funding_fee': 'funding fee at each timestamp', 'open_interest': 'open interest value at each timestamp'}
                         OPEN      HIGH       LOW     CLOSE    VOLUME  \
close_time                                                              
2022-01-02 00:00:00  47681.00  47800.00  47649.25  47704.35  1285.763   
2022-01-02 00:05:00  47704.35  47745.61  47630.09  47660.05   872.760   
2022-01-02 00:10:00  47660.04  47660.05  47530.06  47570.01   661.000   
2022-01-02 00:15:00  47570.01  47570.01  47460.00  47501.49   720.028   
2022-01-02 00:20:00  47501.49  47508.85  47414.86  47415.20   492.074   

                     TAKER_BUY_VOLUME  TAKER_SELL_VOLUME  \
close_time                                                 
2022-01-02 00:00:00           624.229           

## 49. prompt 변수 치환 helper 예시

이 셀은 `load_prompt`, `render_prompt`, `load_and_render_prompt`를 사용해서 evaluator용 user prompt 템플릿에 실제 변수들을 채워 넣는 예시다.

In [1]:
from hypoevolve import load_prompt
from hypoevolve.prompts import render_prompt, load_and_render_prompt
from elg import AtomicNode, Hypothesis, LogicalNode, RelationNode, render_pretty

example_hypothesis = Hypothesis(
    root=RelationNode(
        "IMPLIES",
        [
            LogicalNode("AND", [
                AtomicNode("BTCUSDT_NEW_LOW_SIGNAL_10D == True"),
                AtomicNode("BTCUSDT_NORMALIZED_CLOSE_MOMENTUM_10D < -2.0"),
            ]),
            AtomicNode("ETHUSDT_TRANSFORMED_HIGH_JUMP_10D > 1.5"),
        ],
    )
)

variables = {
    "HYPOTHESIS_PRETTY": render_pretty(example_hypothesis),
    "HYPOTHESIS_JSON": example_hypothesis.to_dict(),
    "DATASET_DESCRIPTION": "Per-asset OHLCV parquet dataset",
    "TIME_COLUMN": "timestamp",
    "ENTITIES": "BTCUSDT, ETHUSDT",
    "COLUMN_SPECS": "- close: close price - high: high price - funding_fee: funding fee",
    "DATASET_ACCESSOR_DOC": "accessor.load_dataframe(entity), accessor.head(entity, n=5), accessor.summary()",
    "DATASET_SAMPLES": "BTCUSDT.head(5), ETHUSDT.head(5)",
}

## 50. 템플릿 파일 로드 후 직접 렌더링하기

먼저 템플릿을 직접 로드하고 `render_prompt()`로 치환하는 방식이다.

In [2]:
user_template = load_prompt("evaluator", "user.md")
rendered_user_prompt = render_prompt(user_template, variables)
print(rendered_user_prompt)

# Hypothesis (Readable)

IMPLIES(
  AND(
    BTCUSDT_NEW_LOW_SIGNAL_10D == True,
    BTCUSDT_NORMALIZED_CLOSE_MOMENTUM_10D < -2.0
  ),
  ETHUSDT_TRANSFORMED_HIGH_JUMP_10D > 1.5
)

# Hypothesis (ELG JSON)

{'root': {'kind': 'relation', 'type': 'IMPLIES', 'inputs': [{'kind': 'logical', 'op': 'AND', 'inputs': [{'kind': 'atomic', 'name': 'BTCUSDT_NEW_LOW_SIGNAL_10D == True', 'type': 'abstract', 'source': 'semantic', 'params': {}}, {'kind': 'atomic', 'name': 'BTCUSDT_NORMALIZED_CLOSE_MOMENTUM_10D < -2.0', 'type': 'abstract', 'source': 'semantic', 'params': {}}], 'params': {}}, {'kind': 'atomic', 'name': 'ETHUSDT_TRANSFORMED_HIGH_JUMP_10D > 1.5', 'type': 'abstract', 'source': 'semantic', 'params': {}}], 'params': {}}, 'params': {}}

# Dataset Description

Per-asset OHLCV parquet dataset

# Time Column

timestamp

# Entities

BTCUSDT, ETHUSDT

# Column Specifications

- close: close price - high: high price - funding_fee: funding fee

# DatasetAccessor Interface

accessor.load_dataframe(entit

## 51. load_and_render_prompt 한 번에 쓰기

파일 로드와 변수 치환을 한 번에 수행하는 방식이다.

In [3]:
rendered_user_prompt_2 = load_and_render_prompt(
    "evaluator",
    "user.md",
    variables=variables,
)
print(rendered_user_prompt_2)

# Hypothesis (Readable)

IMPLIES(
  AND(
    BTCUSDT_NEW_LOW_SIGNAL_10D == True,
    BTCUSDT_NORMALIZED_CLOSE_MOMENTUM_10D < -2.0
  ),
  ETHUSDT_TRANSFORMED_HIGH_JUMP_10D > 1.5
)

# Hypothesis (ELG JSON)

{'root': {'kind': 'relation', 'type': 'IMPLIES', 'inputs': [{'kind': 'logical', 'op': 'AND', 'inputs': [{'kind': 'atomic', 'name': 'BTCUSDT_NEW_LOW_SIGNAL_10D == True', 'type': 'abstract', 'source': 'semantic', 'params': {}}, {'kind': 'atomic', 'name': 'BTCUSDT_NORMALIZED_CLOSE_MOMENTUM_10D < -2.0', 'type': 'abstract', 'source': 'semantic', 'params': {}}], 'params': {}}, {'kind': 'atomic', 'name': 'ETHUSDT_TRANSFORMED_HIGH_JUMP_10D > 1.5', 'type': 'abstract', 'source': 'semantic', 'params': {}}], 'params': {}}, 'params': {}}

# Dataset Description

Per-asset OHLCV parquet dataset

# Time Column

timestamp

# Entities

BTCUSDT, ETHUSDT

# Column Specifications

- close: close price - high: high price - funding_fee: funding fee

# DatasetAccessor Interface

accessor.load_dataframe(entit

## 52. evaluator code generation inference 예시

이 셀은 `prompts/evaluator/system.md`와 `prompts/evaluator/user.md`를 로드하고,
실제 변수들을 채운 뒤 `LLMClient.generate_text()`를 호출해서 **evaluator용 Python 코드 초안**을 받아보는 예시다.

In [3]:
from hypoevolve import load_prompt, load_dataset_schema
from hypoevolve.prompts import load_and_render_prompt
from elg import AtomicNode, Hypothesis, LogicalNode, RelationNode, render_pretty
import json

schema = load_dataset_schema("dataset.yaml")
accessor_doc = """
DatasetAccessor methods:
- accessor.entities() -> list[str]
- accessor.column_names() -> list[str]
- accessor.column_descriptions() -> dict[str, str]
- accessor.load_dataframe(entity) -> pandas.DataFrame
- accessor.load_all_dataframes() -> dict[str, pandas.DataFrame]
- accessor.head(entity, n=5) -> pandas.DataFrame
- accessor.summary() -> dict
""".strip()

measurable_hypothesis = Hypothesis(
    root=RelationNode(
        "IMPLIES",
        [
            LogicalNode(
                "AND",
                [
                    AtomicNode("BTCUSDT_NEW_LOW_SIGNAL_10D@t == True", type="boolean", source="primitive"),
                    AtomicNode("BTCUSDT_NORMALIZED_CLOSE_MOMENTUM_10D@t < -2.0", type="boolean", source="primitive"),
                ],
            ),
            AtomicNode("ETHUSDT_TRANSFORMED_HIGH_JUMP_10D@t+1 > 1.5", type="boolean", source="primitive"),
        ],
    )
)

sample_data = {
    entity: accessor.head(entity, 3).to_dict(orient="records")
    for entity, accessor in [("BTCUSDT", __import__("hypoevolve").dataset.DatasetAccessor(schema)), ("DOGEUSDT", __import__("hypoevolve").dataset.DatasetAccessor(schema)), ("XRPUSDT", __import__("hypoevolve").dataset.DatasetAccessor(schema))]
}

variables = {
    "HYPOTHESIS_PRETTY": render_pretty(measurable_hypothesis),
    "DATASET_DESCRIPTION": schema.description or "",
    "INDEX_NAME": schema.index.name or "",
    "INDEX_DTYPE": schema.index.dtype or "",
    "ENTITIES": ", ".join([f.entity for f in schema.files]),
    "COLUMN_SPECS": "".join(f"- {c.name}: {c.description or ''}" for c in schema.columns),
    "DATASET_ACCESSOR_DOC": accessor_doc,
}

system_prompt = load_prompt("evaluator", "system.md")
user_prompt = load_and_render_prompt("evaluator", "user.md", variables=variables)

print("=== SYSTEM PROMPT ===")
print(system_prompt)
print("=== USER PROMPT (TRUNCATED) ===")
print(user_prompt[:2500])

=== SYSTEM PROMPT ===
# Role

You write Python code that evaluates a measurable ELG hypothesis on the provided dataset.

# Goal

Generate a compact self-contained Python script that defines exactly one function and this function should:

1. load the required data using the provided `DatasetAccessor`
2. compute the probablity of condition event
3. compute the probablity of target event
4. apply the scoring formulation
5. return one Python dictionary containing the required output fields

# Function Contract

The generated code must define this exact function name and signature:

```python
def evaluate_hypothesis(accessor, parameters: dict[str, object] | None = None) -> dict[str, object]:
    ...
```

## Input arguments

### `accessor`
A dataset accessor object will be provided at runtime.
Its available interface and dataset-specific context are described in the user prompt.

### `parameters`
A dictionary of evaluator parameters.
If `parameters is None`, the function should create a defa

## 53. 실제 evaluator code generation 호출

이 셀은 위에서 만든 prompt를 그대로 `LLMClient`에 넣고, 반환된 Python 코드를 출력하는 예시다.

In [4]:
print(variables["HYPOTHESIS_PRETTY"])

IMPLIES(
  AND(
    BTCUSDT_NEW_LOW_SIGNAL_10D@t == True,
    BTCUSDT_NORMALIZED_CLOSE_MOMENTUM_10D@t < -2.0
  ),
  ETHUSDT_TRANSFORMED_HIGH_JUMP_10D@t+1 > 1.5
)


In [5]:
generated_code = client.generate_text(system_prompt, user_prompt)
print(generated_code)

```python
import pandas as pd
import numpy as np

def evaluate_hypothesis(accessor, parameters: dict[str, object] | None = None) -> dict[str, object]:
    if parameters is None:
        parameters = {
            "window_days": 10,
            "momentum_threshold": -2.0,
            "jump_threshold": 1.5,
            "horizon": 1
        }
    
    # Load BTCUSDT and ETHUSDT data
    all_data = accessor.load_all_dataframes()
    btc_df = all_data.get("BTCUSDT")
    eth_df = all_data.get("ETHUSDT")
    
    if btc_df is None or eth_df is None:
        return {
            "combined_score": 0.0,
            "precision": 0.0,
            "baseline": 0.0,
            "coverage": 0.0,
            "uplift": 0.0,
            "support_count": 0,
            "total_count": 0,
            "rationale": "Required BTCUSDT or ETHUSDT data not available."
        }
    
    # Ensure data is sorted by time
    btc_df = btc_df.sort_index()
    eth_df = eth_df.sort_index()
    
    # Align timestamps
  

## 54. LLMEvaluator 사용 예시


In [1]:
from hypoevolve import LLMClient, LLMEvaluator, load_config, load_dataset_schema
from elg import AtomicNode, Hypothesis, LogicalNode, RelationNode

config = load_config("hypoevolve.yaml")
client = LLMClient(config.llm)
schema = load_dataset_schema("dataset.yaml")

hypothesis = Hypothesis(
    root=RelationNode(
        "IMPLIES",
        [
            LogicalNode(
                "AND",
                [
                    AtomicNode("BTCUSDT_NEW_LOW_SIGNAL_10@t == True", type="boolean", source="primitive"),
                    AtomicNode("BTCUSDT_ZSCORE_CLOSE_MOMENTUM_10@t < -2.0", type="boolean", source="primitive"),
                ],
            ),
            AtomicNode("DOGEUSDT_ZSCORE_HIGH_JUMP_10@t+1 > 1.5", type="boolean", source="primitive"),
        ],
    )
)

evaluator = LLMEvaluator(
    llm_client=client,
    dataset_schema=schema,
    dataset_schema_path="dataset.yaml",
)

metrics = evaluator.evaluate(hypothesis)
metrics

{'combined_score': -0.000455706908008132,
 'precision': 0.021628189550425274,
 'baseline': 0.06822524526579968,
 'coverage': 0.009779736101604685,
 'uplift': -0.046597055715374404,
 'support_count': 4115,
 'total_count': 420768,
 'rationale': 'Condition occurs 4115/420768 times. When condition is true, target occurs 2.2% vs baseline 6.8%. Uplift -0.047 weighted by coverage 0.010.'}

## 55. mutation steering 사용 예시

이 예시는 현재 mutation steering이 **candidate index를 고르는 방식이 아니라**, parent measurable ELG와 현재 metric 문맥을 바탕으로 **새 child ELG를 직접 생성하는 방식**으로 동작하는 모습을 보여준다.

출력은 full child ELG이지만, 프롬프트는 여전히 `replace_atomic`, `change_logical_operator`, `append_child` 같은 **local mutation 스타일**을 따르도록 유도한다. 따라서 결과를 읽을 때는 `mutation_summary`를 통해 parent 대비 어떤 국소 변경이 적용되었는지 함께 확인하면 된다.


In [1]:
from hypoevolve import LLMClient, load_config, steer_mutation
from hypoevolve.parser import ParseError
from elg import AtomicNode, Hypothesis, LogicalNode, RelationNode, render_pretty

config = load_config("hypoevolve.yaml")
client = LLMClient(config.llm)

parent_hypothesis = Hypothesis(
    root=RelationNode(
        "IMPLIES",
        [
            LogicalNode(
                "AND",
                [
                    AtomicNode("BTCUSDT_NEW_LOW_SIGNAL_W12@t == True", type="boolean", source="primitive"),
                    AtomicNode("BTCUSDT_ZSCORE_CLOSE_MOMENTUM_W12@t < -1.5", type="boolean", source="primitive"),
                ],
            ),
            AtomicNode("DOGEUSDT_ZSCORE_HIGH_JUMP_W12@t+1 > 1.5", type="boolean", source="primitive"),
        ],
    )
)

current_metrics = {
    "combined_score": -0.00045,
    "precision": 0.0216,
    "baseline": 0.0682,
    "coverage": 0.0098,
    "uplift": -0.0466,
    "support_count": 4115,
    "total_count": 420768,
    "rationale": "Condition occurs 4115/420768 times. When condition is true, target occurs 2.2% vs baseline 6.8%."
}

recent_history = [
    {
        "mutation_summary": "Applied a replace_atomic-style local mutation on the condition side by changing BTCUSDT_ZSCORE_CLOSE_MOMENTUM_W12@t < -1.5 to BTCUSDT_ZSCORE_CLOSE_MOMENTUM_W12@t < -2.0.",
        "result_hypothesis": "IMPLIES(AND(BTCUSDT_NEW_LOW_SIGNAL_W12@t == True, BTCUSDT_ZSCORE_CLOSE_MOMENTUM_W12@t < -2.0), DOGEUSDT_ZSCORE_HIGH_JUMP_W12@t+1 > 1.5)",
        "note": "Made the condition-side momentum threshold stricter."
    },
    {
        "mutation_summary": "Applied a replace_atomic-style local mutation on the condition side by changing BTCUSDT_ZSCORE_CLOSE_MOMENTUM_W12@t < -1.5 to BTCUSDT_ZSCORE_CLOSE_MOMENTUM_W12@t < -1.0.",
        "result_hypothesis": "IMPLIES(AND(BTCUSDT_NEW_LOW_SIGNAL_W12@t == True, BTCUSDT_ZSCORE_CLOSE_MOMENTUM_W12@t < -1.0), DOGEUSDT_ZSCORE_HIGH_JUMP_W12@t+1 > 1.5)",
        "note": "Made the condition-side momentum threshold looser."
    },
]

try:
    decision = steer_mutation(
        parent_hypothesis=parent_hypothesis,
        parent_hypothesis_nl="If BTCUSDT has a new low signal and its close-momentum z-score is very negative, then DOGEUSDT will show a strong high-jump z-score on the next step.",
        current_metrics=current_metrics,
        llm=client,
        atomic_pool=[
            AtomicNode("DOGEUSDT_ZSCORE_HIGH_JUMP_W12@t+1 > 1.0", type="boolean", source="primitive"),
            AtomicNode("BTCUSDT_ZSCORE_CLOSE_MOMENTUM_W12@t < -1.5", type="boolean", source="primitive"),
        ],
        recent_history=recent_history,
        top_hypotheses=[],
        retries=2,
    )
except ParseError as exc:
    print("steering failed:", exc)
    if exc.errors:
        print("attempt details:")
        for item in exc.errors:
            print("-", item)
    raise

print("reason:", decision.reason)
print("mutation_summary:", decision.mutation_summary)
print("child hypothesis:")
print(render_pretty(decision.child_hypothesis))

2026-03-30 06:03:00.946 | INFO     | hypoevolve.mutation:steer_mutation:94 - steering proposed child hypothesis successfully


reason: The current hypothesis has a negative uplift (-0.0466) because precision (2.2%) is significantly below baseline (6.8%). This suggests the immediate next step (t+1) target is not correlated with the BTC low-signal condition. In cryptocurrency markets, lagged reactions and cross-asset sentiment propagation are common. The 'meme coin' DOGE often reacts with a slight delay to BTC movements, especially after extreme signals. Recent mutation history shows adjusting the momentum threshold didn't resolve the core issue. Changing the target time horizon from t+1 to t+2 tests whether DOGE's high-jump response occurs with a 2-step delay rather than immediately. This preserves the measurable structure while addressing the temporal misalignment. The coverage (0.98%) is already low but reasonable for rare signals; this mutation doesn't further reduce it. If the precision improves toward or above baseline at t+2, uplift and combined_score will increase. This is a minimal, interpretable change

## 56. random exploration mutation steering 사용 예시

이 예시는 score를 직접 최적화하지 않고, 최근 mutation history와 다른 방향의 local mutation을 생성하는 exploration용 steering 프롬프트 사용 예시다.


In [1]:
from hypoevolve import LLMClient, load_config
from hypoevolve.helper import build_steering_prompt_variables
from hypoevolve.prompts import load_and_render_prompt, load_prompt
from elg import AtomicNode, Hypothesis, LogicalNode, RelationNode, hypothesis_from_dict, render_pretty
import json

config = load_config("hypoevolve.yaml")
client = LLMClient(config.llm)

parent_hypothesis = Hypothesis(
    root=RelationNode(
        "IMPLIES",
        [
            LogicalNode(
                "AND",
                [
                    AtomicNode("BTCUSDT_NEW_LOW_SIGNAL_W12@t == True", type="boolean", source="primitive"),
                    AtomicNode("BTCUSDT_ZSCORE_CLOSE_MOMENTUM_W12@t < -1.5", type="boolean", source="primitive"),
                ],
            ),
            AtomicNode("DOGEUSDT_ZSCORE_HIGH_JUMP_W12@t+1 > 1.5", type="boolean", source="primitive"),
        ],
    )
)

recent_history = [
    {
        "mutation_summary": "Applied a replace_atomic_threshold-style local mutation by changing BTCUSDT_ZSCORE_CLOSE_MOMENTUM_W12@t < -1.5 to < -2.0.",
        "result_hypothesis": "IMPLIES(AND(BTCUSDT_NEW_LOW_SIGNAL_W12@t == True, BTCUSDT_ZSCORE_CLOSE_MOMENTUM_W12@t < -2.0), DOGEUSDT_ZSCORE_HIGH_JUMP_W12@t+1 > 1.5)",
        "note": "A stricter momentum threshold was tried recently."
    },
    {
        "mutation_summary": "Applied a replace_atomic_threshold-style local mutation by changing DOGEUSDT_ZSCORE_HIGH_JUMP_W12@t+1 > 1.5 to > 1.0.",
        "result_hypothesis": "IMPLIES(AND(BTCUSDT_NEW_LOW_SIGNAL_W12@t == True, BTCUSDT_ZSCORE_CLOSE_MOMENTUM_W12@t < -1.5), DOGEUSDT_ZSCORE_HIGH_JUMP_W12@t+1 > 1.0)",
        "note": "A looser target threshold was also tried recently."
    },
]

variables = build_steering_prompt_variables(
    parent_hypothesis=parent_hypothesis,
    parent_hypothesis_nl="If BTCUSDT has a new low signal and its close-momentum z-score is very negative, then DOGEUSDT will show a strong high-jump z-score on the next step.",
    current_metrics={},
    recent_history=recent_history,
    top_hypotheses=[],
)

system_prompt = load_prompt("steering-random", "system.md")
user_prompt = load_and_render_prompt("steering-random", "user.md", variables=variables)

payload = client.generate_json(system_prompt, user_prompt)
child_hypothesis = hypothesis_from_dict({"root": payload["child_hypothesis"], "params": {}})

print("mutation_summary:", payload["mutation_summary"])
print("child_hypothesis instance:")
print(render_pretty(child_hypothesis))

mutation_summary: Applied 3 local mutations: 1) change_relation_type from IMPLIES to SUPPORT, 2) replace_atomic_threshold on BTCUSDT_ZSCORE_CLOSE_MOMENTUM_W12@t from <-1.5 to <-2.0, 3) wrap_not on target node DOGEUSDT_ZSCORE_HIGH_JUMP_W12@t+1 > 1.5.
child_hypothesis instance:
SUPPORT(
  AND(
    BTCUSDT_NEW_LOW_SIGNAL_W12@t == True,
    BTCUSDT_ZSCORE_CLOSE_MOMENTUM_W12@t < -2.0
  ),
  NOT(
    DOGEUSDT_ZSCORE_HIGH_JUMP_W12@t+1 > 1.5
  )
)
